## BÀI TẬP VỀ NHÀ

- **Họ và tên:** Phan Đăng Trọng Tín
- **MSSV:** 25521866

1. Khai báo thư viện

In [3]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
import sys
!{sys.executable} -m pip install openpyxl

pd.set_option("display.max_columns", None)

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]




[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


2. Đọc và tiền xử lý dữ liệu gốc
- Đọc dữ liệu trực tiếp từ file Excel gốc trong cùng thư mục. Quy trình này bao gồm các bước: đọc dữ liệu, chuẩn hóa tên các cột, kiểm tra và loại bỏ các dòng bị khuyết thiếu (missing values) cũng như các dòng dữ liệu trùng lặp (duplicate).

In [4]:
DATA_PATH = Path(r"C:\Users\Admin\Desktop\EXAM\mliot-pyml-2026-hw\week04\Homework_b7\Dry_Bean_Dataset\Dry_Bean_Dataset.xlsx")

df = pd.read_excel(DATA_PATH, engine="openpyxl")

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

df = df.dropna().reset_index(drop=True)

df = df.drop_duplicates().reset_index(drop=True)

3. Phân tách tập dữ liệu và lưu CSV

In [5]:
target = "class"

numeric_columns = [col for col in df.columns if col != target]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

df[target] = df[target].astype(str).str.strip().str.upper()

X = df.drop(target, axis=1)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_df = X_train.copy()
train_df[target] = y_train
train_df = train_df.reset_index(drop=True)

test_df = X_test.copy()
test_df[target] = y_test
test_df = test_df.reset_index(drop=True)

train_df.to_csv("dry_bean_train.csv", index=False)
test_df.to_csv("dry_bean_test.csv", index=False)

4. Tải dữ liệu từ CSV và tách đặc trưng

In [6]:
train_bean = pd.read_csv("dry_bean_train.csv")
test_bean = pd.read_csv("dry_bean_test.csv")

X_train_b = train_bean.drop("class", axis=1)
y_train_b = train_bean["class"]

X_test_b = test_bean.drop("class", axis=1)
y_test_b = test_bean["class"]

5. Chuẩn hóa dữ liệu
- Khởi tạo công cụ StandardScaler và tiến hành chuẩn hóa. Tham số chỉ được fit trên tập dữ liệu huấn luyện, sau đó áp dụng biến đổi (transform) cho cả tập huấn luyện và tập kiểm tra.

In [7]:
scaler = StandardScaler()

X_train_b_scaled = scaler.fit_transform(X_train_b)

X_test_b_scaled = scaler.transform(X_test_b)

6. Huấn luyện và đánh giá mô hình Logistic Regression
- Bài toán phân loại hạt dưa có 7 lớp đầu ra khác nhau. Mô hình Logistic Regression được khởi tạo với cấu hình multi_class='multinomial' và số vòng lặp tối đa được tăng lên 2000 để mô hình dễ dàng hội tụ.

In [8]:
lr_bean = LogisticRegression(max_iter=2000, multi_class='multinomial')
lr_bean.fit(X_train_b_scaled, y_train_b)

y_pred_lr_bean = lr_bean.predict(X_test_b_scaled)

print("Logistic Regression - Classification Report:")
print(classification_report(y_test_b, y_pred_lr_bean))

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Logistic Regression - Classification Report:
              precision    recall  f1-score   support

    BARBUNYA       0.93      0.89      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.91      0.94      0.93       326
    DERMASON       0.93      0.91      0.92       709
       HOROZ       0.96      0.94      0.95       372
       SEKER       0.93      0.94      0.94       406
        SIRA       0.86      0.88      0.87       527

    accuracy                           0.92      2709
   macro avg       0.93      0.93      0.93      2709
weighted avg       0.92      0.92      0.92      2709



7. Huấn luyện và đánh giá mô hình K-Nearest Neighbors
- Áp dụng thuật toán KNN với số điểm lân cận bằng 5 (n_neighbors=5), huấn luyện mô hình dựa trên tập dữ liệu đã chuẩn hóa và đánh giá qua báo cáo phân loại (Classification Report).

In [9]:
knn_bean = KNeighborsClassifier(n_neighbors=5)
knn_bean.fit(X_train_b_scaled, y_train_b)

y_pred_knn_bean = knn_bean.predict(X_test_b_scaled)

print("K-Nearest Neighbors - Classification Report:")
print(classification_report(y_test_b, y_pred_knn_bean))

K-Nearest Neighbors - Classification Report:
              precision    recall  f1-score   support

    BARBUNYA       0.93      0.88      0.90       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.90      0.95      0.92       326
    DERMASON       0.92      0.91      0.91       709
       HOROZ       0.96      0.93      0.95       372
       SEKER       0.95      0.94      0.94       406
        SIRA       0.85      0.87      0.86       527

    accuracy                           0.92      2709
   macro avg       0.93      0.93      0.93      2709
weighted avg       0.92      0.92      0.92      2709

